# PFN quality: `pfn_large` vs the smoke PFN

## Description

**Question.** How good is `pfn_large`'s posterior predictive distribution (PPD) on held-out draws of
its training prior, per input dimension d and context size n, compared with the smoke PFN, and what does
a forward pass cost? This is also the port's **sanity check**: the evaluation was run before in
pfn-acqpolicy (`eval_pfn_large`, commit 58b6e2b), on the same draws.

**Setup.** Two registered checkpoints (`pfn_large`: 6.6M params, 1000 bins, trained on n ≤ 100;
`pfn_smoke`: 139k params, 64 bins, n ≤ 20) on 256 held-out BNN-prior functions per d = 1…6 (data seed
555,000 + d), 200 test points each, nested context prefixes n ∈ {1, …, 100}. Metrics per test point
(NLL incl. log bin width, squared error of the mean, CRPS, PIT, 50/90% coverage, predictive std) and
per function (Spearman of mean vs y, regret of picking the argmin of the mean); 95% bootstrap CIs over
functions. CPU forward cost (8 threads). 1-d panels: the PPD density on the bar intervals, mean, 5–95%
band, true function and context, for the three draws with the most local minima.

**What to look for.** Lower NLL/CRPS for `pfn_large` at every d and n; 90% coverage ≈ 0.9 (calibration);
monotone improvement in n; the smoke model's degradation beyond n = 20 (outside its training range);
in 1-d: the band collapsing onto the data near context points and widening away from them, the mean
tracking local minima once points are near them. Failure modes to spot at a glance: a band that does
not contain the true function, multimodal density far from data that ignores the context, a mean that
interpolates flat lines between points.

**Reproduced / not reproduced.** Reproduced if every metric in `metrics` matches the original run to
within 1e-5 (same draws, same code apart from import paths, CPU); otherwise the port changed behaviour
and the difference is explained before anything else is built on it. Timing is machine-dependent and
not part of the check.

**Limits.** In-distribution only (the BNN training prior, d ≤ 6, n ≤ 100); says nothing about real
benchmarks or about BO performance.

In [ ]:
from pfn_repro.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/eval_pfn_large/<sweep_id>"  # set after `just snapshot runs/eval_pfn_large/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.